# 🎧 OmniVoice-Thai — โคลนเสียงตัวเอง + แก้จุดอ่อนอ่านตัวเลข

notebook นี้ทำ 2 อย่าง:
1. **โคลนเสียงคุณ** — ลำดับการหาเสียงอ้างอิง: ① ไฟล์ใน repo (`tts-ai/tests/ref/my_voice.*` ที่ Claude commit ให้) → ② ไฟล์ที่คุณอัปโหลดเข้า Colab เอง (`my_voice.wav/mp3/m4a` + พิมพ์ข้อความในเซลล์ 3) → ③ ถ้าไม่มีทั้งคู่ ใช้เสียงเปรมวดีไปก่อน
2. **แปลงตัวเลขเป็นคำอ่านอัตโนมัติ** (normalize_th.py) — มีคู่เทียบประโยคตัวเลขแบบแปลง vs ไม่แปลง ให้ฟังความต่างชัด ๆ

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** → Save → Runtime → **Run all** → ฟังที่เซลล์ 6

In [ ]:
# 1) ติดตั้ง (~2 นาที)
!pip install -q omnivoice edge-tts soundfile pythainlp
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี — แนะนำเปิด T4')

In [ ]:
# 2) ประโยคทดสอบ + ตัวแปลงตัวเลขเป็นคำอ่าน (ดึงจาก repo)
import os, time, requests
os.makedirs('samples_myvoice', exist_ok=True)

RAW_BASE = 'https://raw.githubusercontent.com/suthipongs10-a11y/claude-project1/claude/tts-ai-tools-resources-m5rku5/tts-ai'
r = requests.get(f'{RAW_BASE}/normalize_th.py', timeout=30)
open('normalize_th.py', 'w', encoding='utf-8').write(r.text)
from normalize_th import normalize_th

SENTENCES = {
  '01_basic': 'สวัสดีครับ ยินดีต้อนรับเข้าสู่ช่องของเรา วันนี้มีเรื่องราวน่าสนใจมาเล่าให้ฟังกันครับ',
  '02_story': 'ลองจินตนาการดูนะครับ ชายคนหนึ่งตัดสินใจเดินออกจากช่องยูทูบที่มีผู้ติดตามยี่สิบล้านคน โดยไม่บอกใครแม้แต่คำเดียว',
  '03_numbers_raw': 'เมื่อเวลา 18:30 น. ของวันที่ 2 สิงหาคม 2569 ยอดวิวพุ่งไปถึง 2,500,000 วิว สร้างรายได้กว่า 64,000 บาท',
  '04_english_mix': 'ช่อง YouTube ของเราใช้ AI ทำคลิป Shorts ลง TikTok และ Facebook ทุกวัน',
  '05_english_thai_spelled': 'ช่องยูทูบของเราใช้เอไอทำคลิปช็อตส์ ลงติ๊กต่อกและเฟซบุ๊กทุกวัน',
  '06_emotion': 'อะไรนะครับ! เขาทิ้งรายได้เดือนละหลายล้านบาทไปเฉย ๆ อย่างนั้นเหรอ เรื่องนี้มันบ้ามาก ๆ ไม่มีใครเชื่อเลยจริง ๆ',
}
timing = {}

print('ตัวอย่างการแปลงประโยคตัวเลข:')
print('ก่อน:', SENTENCES['03_numbers_raw'])
print('หลัง:', normalize_th(SENTENCES['03_numbers_raw']))

In [ ]:
# 3) หาเสียงอ้างอิง: repo -> ไฟล์อัปโหลดเอง -> เปรมวดี (fallback)
MY_VOICE_TEXT = ''  # <-- ถ้าอัปโหลดไฟล์เอง (my_voice.wav/mp3/m4a) พิมพ์ข้อความที่พูดในไฟล์ตรงนี้ให้เป๊ะ

REF_AUDIO = REF_TEXT = src = None

# ① ไฟล์ใน repo (Claude commit ให้ที่ tts-ai/tests/ref/)
for ext in ('wav', 'mp3', 'm4a'):
    r = requests.get(f'{RAW_BASE}/tests/ref/my_voice.{ext}', timeout=30)
    if r.status_code == 200:
        open(f'my_voice.{ext}', 'wb').write(r.content)
        t = requests.get(f'{RAW_BASE}/tests/ref/my_voice.txt', timeout=30)
        if t.status_code == 200 and t.text.strip():
            REF_AUDIO, REF_TEXT, src = f'my_voice.{ext}', t.text.strip(), 'เสียงจริงจาก repo ✅'
        break

# ② ไฟล์ที่อัปโหลดเข้า Colab เอง (ลากไฟล์ใส่แถบ Files 📁 ซ้ายมือ)
if not REF_AUDIO:
    for ext in ('wav', 'mp3', 'm4a'):
        if os.path.exists(f'my_voice.{ext}') and MY_VOICE_TEXT.strip():
            REF_AUDIO, REF_TEXT, src = f'my_voice.{ext}', MY_VOICE_TEXT.strip(), 'ไฟล์อัปโหลดเอง ✅'
            break

# ③ ยังไม่มีเสียงจริง -> ใช้เปรมวดีไปก่อน
if not REF_AUDIO:
    import edge_tts
    await edge_tts.Communicate(SENTENCES['01_basic'], 'th-TH-PremwadeeNeural').save('ref_premwadee.mp3')
    REF_AUDIO, REF_TEXT, src = 'ref_premwadee.mp3', SENTENCES['01_basic'], 'เปรมวดี (ยังไม่มีคลิปเสียงจริง)'

!ffmpeg -y -loglevel error -i {REF_AUDIO} -ar 24000 -ac 1 ref_24k.wav
REF_AUDIO = 'ref_24k.wav'
print('เสียงอ้างอิง:', src)
print('ข้อความอ้างอิง:', REF_TEXT)

In [ ]:
# 4) โหลด OmniVoice-Thai (~3-4 นาทีครั้งแรก)
assert 'SENTENCES' in globals(), '⚠️ ยังไม่ได้รันเซลล์ 2'
import soundfile as sf
from omnivoice import OmniVoice

dev = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dt = torch.float16 if torch.cuda.is_available() else torch.float32
MODEL_ID = 'hotdogs/omnivoice-thai'
try:
    model = OmniVoice.from_pretrained(MODEL_ID, device_map=dev, dtype=dt)
except Exception as e:
    print('โหลดตัวไฟน์จูนไทยไม่ได้ ->', e, '\nใช้ตัวฐานแทน')
    MODEL_ID = 'k2-fsa/OmniVoice'
    model = OmniVoice.from_pretrained(MODEL_ID, device_map=dev, dtype=dt)
print('ใช้โมเดล:', MODEL_ID)

In [ ]:
# 5) เจนเสียง — ทุกประโยคผ่าน normalize_th ก่อน + แถมประโยคตัวเลขแบบดิบไว้ฟังเทียบ
GEN = {sid: normalize_th(text) for sid, text in SENTENCES.items()}
GEN['03z_numbers_no_normalize'] = SENTENCES['03_numbers_raw']  # แบบไม่แปลง (คู่เทียบ)

for sid, text in GEN.items():
    try:
        t0 = time.time()
        audio = model.generate(text=text, ref_audio=REF_AUDIO, ref_text=REF_TEXT)
        sf.write(f'samples_myvoice/omni_{sid}.wav', audio[0], 24000)
        timing[f'omni_{sid}'] = time.time() - t0
        print(f'เสร็จ: {sid} ({timing[f"omni_{sid}"]:.1f} วิ)')
    except Exception as e:
        print('ข้าม', sid, '->', e)

In [ ]:
# 6) 🎧 ฟัง — เสียงอ้างอิงบนสุด แล้วไล่ทีละประโยค (03 มีคู่เทียบ แปลง/ไม่แปลง)
import glob
from IPython.display import Audio, display, Markdown

display(Markdown(f'### เสียงอ้างอิงที่ใช้โคลน ({src})'))
display(Audio(REF_AUDIO))

for sid, text in GEN.items():
    f = f'samples_myvoice/omni_{sid}.wav'
    if not os.path.exists(f) or os.path.getsize(f) == 0:
        continue
    secs = timing.get(f'omni_{sid}')
    display(Markdown(f'---\n### {sid}' + (f' — {secs:.1f} วิ' if secs else '') + f'\n> ข้อความที่ส่งเข้าโมเดล: {text}'))
    display(Audio(f))

In [ ]:
# 7) (ทางเลือก) โหลดเก็บเป็น zip
!zip -q -r omni_myvoice_samples.zip samples_myvoice
try:
    from google.colab import files
    files.download('omni_myvoice_samples.zip')
except Exception as e:
    print('ดาวน์โหลดอัตโนมัติไม่ได้:', e)
    print('ดึงเองได้ที่แถบ Files 📁 ซ้ายมือ')